In [ ]:
#@title Preparar entorno e interactividad { display-mode: "form" }
import json, html as html_lib, base64
from IPython.display import display, HTML

def tutorial(url, alto=760, titulo="Tutorial"):
    sep = "&" if "?" in url else "?"
    src = f"{url}{sep}embed=1"
    display(HTML(f'''
    <iframe src="{src}" width="100%" height="{alto}"
            style="border:0;display:block;border-radius:10px;background:#faf7f0;"
            loading="lazy" title="{html_lib.escape(titulo)}"></iframe>
    <p style="margin:8px 0 0;">
      <a href="{url}" target="_blank" rel="noopener">
        Abrir {html_lib.escape(titulo)} en pantalla completa ↗
      </a>
    </p>
    '''))

def pregunta_interactiva(numero, tema, pregunta, opciones, correcta, retro):
    uid = f"s05-p{numero}"
    opts = "".join(
        f'<label style="display:block;margin:8px 0;"><input type="radio" name="{uid}" value="{i}"> '
        f'{html_lib.escape(op)}</label>'
        for i, op in enumerate(opciones)
    )
    retro_json = json.dumps(retro, ensure_ascii=False)
    display(HTML(f'''
    <div style="border:2px solid #175c3c;border-radius:12px;padding:16px;margin:14px 0;background:#f4faf6;color:#172019;">
      <div style="font-weight:700;color:#123f2b;margin-bottom:8px;">Pregunta {numero} · {html_lib.escape(tema)}</div>
      <p><strong>{html_lib.escape(pregunta)}</strong></p>
      {opts}
      <button onclick="(function(){{
          const e=document.querySelector('input[name={uid}]:checked');
          const s=document.getElementById('r-{uid}');
          if(!e){{s.innerHTML='Selecciona una opción.';return;}}
          const i=Number(e.value); const r={retro_json};
          const ok=i==={correcta};
          s.innerHTML='<div style=&quot;margin-top:10px;padding:10px;border-radius:8px;background:'+
            (ok?'#d1e7dd;color:#0f5132':'#f8d7da;color:#842029')+'&quot;><strong>'+
            (ok?'Correcto. ':'Revisa. ')+'</strong>'+r[i]+'</div>';
        }})()"
        style="background:#175c3c;color:white;border:0;border-radius:7px;padding:8px 13px;cursor:pointer;">
        Verificar
      </button>
      <div id="r-{uid}" aria-live="polite"></div>
    </div>
    '''))
print("Entorno de la sesión 5 listo.")


def pregunta_interactiva_codificada(payload_b64):
    payload = json.loads(base64.b64decode(payload_b64).decode("utf-8"))
    pregunta_interactiva(
        payload["numero"], payload["tema"], payload["pregunta"],
        payload["opciones"], payload["correcta"], payload["retro"]
    )

<a href="https://colab.research.google.com/github/jazaineam1/BigData2026/blob/main/Cuadernos/5_Atlas_Cassandra_Query_First.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir la sesión 5 en Google Colab">
</a>

**Acceso público:** [página del curso](https://jazaineam1.github.io/BigData2026/)

> **Punto de partida real.** La sesión anterior terminó después de crear `compras_claras` en Atlas
> y cargar las colecciones `noticias` y `entidades_noticias`. Hoy continuamos desde ahí:
> no repetimos el registro ni la carga.

# Sesión 5 — De datos persistidos en Atlas a una consulta operacional con Cassandra

## Universidad Central
> ### Facultad de Ingeniería y Ciencias Básicas
> ### Maestría en Analítica de Datos — BIG DATA (64491093)

**Fecha:** 3 de septiembre de 2026  
**Tema ajustado del PDA:** práctica de Cassandra, preservando el cierre pendiente de MongoDB Atlas  
**Caso conductor:** Compras Claras  
**Pregunta profesional:** **¿cómo convierte Laura la evidencia que ya dejó persistida en Atlas en una bandeja priorizada y luego en una consulta que pueda repetir sin reconstruir todo cada vez?**

### CONTRATO DE ÉXITO S05 — tres productos, las mismas herramientas

Hoy seguimos usando **todo** el recorrido técnico del curso:

```text
Colab → MongoDB Atlas → tutorial visual → Colab/pandas → Astra/CQL → Colab/Python
```

La diferencia es que no vas a sentir nueve entregables distintos. Todo queda agrupado en **tres productos**:

| Producto | Qué construyes | Controles/evidencias que viven dentro |
|---|---|---|
| **1 · Vista Atlas** | `menciones_clasificadas` | 142 entidades · 6 alta / 25 media / 111 baja |
| **2 · Bandeja explicable** | 77 candidatos de revisión | `1.000 → 163 → 77` + contraste de hipótesis H1 + límite de la prensa |
| **3 · Consulta operacional** | `corte + departamento → top 5` en Cassandra | diseño query-first + CQL + CRUD/automatización desde Python + comparación con pandas |

El **hito descargable** y `s05_ancla_s06.json` son salidas automáticas que documentan esos tres productos; no son productos adicionales.

> **Criterio de éxito.** Si puedes explicar la vista, justificar cómo llegamos a 77 y razonar por qué la tabla Cassandra responde el top 5, completaste el núcleo conceptual. La conexión Python sigue en la sesión, pero su código de infraestructura se **ejecuta**, no se memoriza.

### DEUDA ABIERTA S04 — la clase terminó con datos, no con una decisión

La sesión 4 alcanzó un punto concreto y útil:

```text
Atlas
└── compras_claras
    ├── noticias               → 987 documentos
    └── entidades_noticias     → 142 documentos
```

Eso resolvió **dónde vive la evidencia y cómo la comparte el equipo**, pero dejó abierta la pregunta que daba nombre a S4:

> **¿Qué contrato debe revisar Laura primero?**

S5 existe para cerrar esa deuda. Primero convierte las 142 entidades en una vista explicable; después cruza esa señal con SECOP para construir la bandeja; solo cuando la bandeja existe aparece Cassandra como problema de servicio repetitivo.

**PARA LLEVAR.** S4 dejó persistencia compartida. S5 debe producir una decisión operacional reproducible.

### CONTINUIDAD S03-S05 — del prototipo a la bandeja operacional

En S3 apareció una primera bandeja de **200 procesos**. Ese resultado fue un **prototipo exploratorio**: servía para demostrar que las señales podían reducir el universo, pero todavía mezclaba decisiones que no habíamos convertido en una regla operacional estable.

Hoy no estamos “corrigiendo 200 por 77” ni comparando el mismo indicador. En S5 fijamos una regla distinta, explícita y reproducible sobre la muestra de 1.000 procesos:

```text
entidad presente en prensa
+ modalidad contiene "directa"
+ respuestas al procedimiento = 0
────────────────────────────────
77 candidatos
```

**PARA LLEVAR.** El número importante no es 77 por sí solo. Lo importante es que Laura puede explicar exactamente **cómo entró cada proceso** y qué evidencia todavía falta.

## Mapa de la sesión — mismas herramientas, menos cambios mentales

| Tramo | Rol | Pregunta | Herramienta | Qué queda |
|---|---|---|---|---|
| 1. Retomar S4 | 🧠 **ENTIENDE** | ¿desde dónde partimos? | Colab + Atlas | 987 + 142 |
| 2. Construir vista | ▶️ **EJECUTA** | ¿cómo publicamos la clasificación? | tutorial + Atlas | producto 1 |
| 3. Construir bandeja | 🧠 + ▶️ | ¿qué procesos pasan primero? | Colab + pandas/SECOP | producto 2 |
| 4. Contrastar hipótesis | 🧠 **ENTIENDE** | ¿qué evidencia aporta realmente la prensa? | Colab | alcance correcto |
| 5. Diseñar Cassandra | 🧠 **ENTIENDE** | ¿qué pregunta repetitiva debemos servir? | papel + cuaderno | partición + orden |
| 6. Implementar CQL | ▶️ **EJECUTA** | ¿la tabla sirve esa pregunta? | tutorial + Astra/CQL | producto 3, tramo A |
| 7. Automatizar | ▶️ + ✏️ **MODIFICA** | ¿cómo usa una aplicación la misma tabla? | Colab + Python | producto 3, tramo B |
| 8. Cerrar | 🧠 **EXPLICA** | ¿qué resolvió cada motor y qué límite queda? | cuaderno | hito + ancla S6 |

### SEMÁFORO DE CÓDIGO S05

- 🧠 **ENTIENDE:** debes poder explicarlo con tus palabras.
- ▶️ **EJECUTA:** corre la celda y verifica la salida; **no necesitas escribirla de memoria**.
- ✏️ **MODIFICA:** cambia únicamente el dato señalado y observa qué ocurre.

**Regla de navegación del curso:** el cuaderno explica **por qué, qué significa y qué límite tiene**; los tutoriales HTML explican **dónde hacer clic y qué debe aparecer**.

---
## 1. Reactivar sin repetir la sesión 4

Antes de entrar a consultas, verifica el estado esperado:

```text
Atlas
└── compras_claras
    ├── noticias               → 987 documentos
    └── entidades_noticias     → 142 documentos
```

La siguiente celda solo recupera la conexión. La contraseña se pide con `getpass()` y no queda escrita.

<details>
<summary><strong>Si no conservas la URI de S4</strong></summary>

1. En Atlas abre tu clúster → **Connect / Drivers**.
2. Selecciona **Python** y copia la URI con `<db_username>` y `<db_password>`.
3. Vuelve aquí. **No crees otra base ni otro clúster.**

Si tampoco recuerdas el camino, usa `assets/tutoriales/atlas-guia-conexion.html` solo como recuperación.
</details>

In [ ]:
!pip install -q pymongo dnspython

from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient

uri_pegada = input("Pega tu URI de Atlas (Connect / Drivers): ").strip()
if not uri_pegada:
    raise ValueError("La URI está vacía. Recupera la URI de tu clúster de S4.")

uri = uri_pegada
if "<db_username>" in uri or "<db_password>" in uri:
    usuario = input("Usuario de base de datos: ").strip()
    contrasena = quote_plus(getpass("Contraseña (no se muestra): "))
    uri = uri.replace("<db_username>", quote_plus(usuario))
    uri = uri.replace("<db_password>", contrasena)

try:
    client = MongoClient(uri, serverSelectionTimeoutMS=7000)
    client.admin.command("ping")
    db = client["compras_claras"]
    motor_atlas = "Atlas real"
    print("Conectado.")
    print("noticias:", db["noticias"].count_documents({}))
    print("entidades_noticias:", db["entidades_noticias"].count_documents({}))
except Exception as error:
    db = None
    motor_atlas = "respaldo por archivos"
    print("No se pudo conectar:", type(error).__name__)
    print("La clase puede continuar con respaldo; eso NO sustituye haber creado la vista en Atlas.")

print("Modo:", motor_atlas)

## 2. De documentos a una vista que sí usaremos

| Objeto | Pregunta mental | ¿Guarda datos nuevos? |
|---|---|---|
| filtro / `find()` | ¿qué documentos quiero ver? | no |
| pipeline | ¿qué transformaciones quiero encadenar? | no |
| pipeline guardado | ¿quiero conservar la receta? | no |
| vista | ¿quiero consultar el resultado como objeto de solo lectura? | no |

La historia necesita una transformación: convertir menciones por entidad en una señal explicable.

| Etapa | Para qué sirve aquí | Qué debes mirar |
|---|---|---|
| `$set` | agrega `nivel_menciones` | crea un campo |
| `$switch` | aplica los cortes 20 y 5 | primera condición verdadera gana |
| `$project` | deja los campos útiles | reduce ruido |
| `$sort` | ordena la salida | la hace legible |

| `noticias` | nivel esperado |
|---:|---|
| 4 | baja |
| 5 | media |
| 19 | media |
| 20 | alta |

**PARA LLEVAR.** El pipeline es la receta; la vista publica esa receta como salida consultable de solo lectura.

### MICROEJEMPLO SWITCH S05 — primero decide, después traduce a MongoDB

Antes de mirar `$switch`, resuelve tres casos sin sintaxis nueva:

| entidad | noticias | nivel |
|---|---:|---|
| Entidad A | 3 | baja |
| Entidad B | 8 | media |
| Entidad C | 24 | alta |

En Python, la misma regla se leería así:

```python
if noticias >= 20:
    nivel = "alta"
elif noticias >= 5:
    nivel = "media"
else:
    nivel = "baja"
```

MongoDB no introduce una lógica diferente: `$switch` expresa esa misma decisión dentro del pipeline. Evalúa las ramas **en orden** y usa la primera condición verdadera.

```text
noticias = 25 → ¿>=20? sí → alta → deja de probar
noticias =  8 → ¿>=20? no → ¿>=5? sí → media
```

**OJO.** Los cortes `5` y `20` son una **regla pedagógica versionada** para resumir intensidad de menciones. No son umbrales oficiales de riesgo ni fueron estimados con un modelo estadístico.

**PARA LLEVAR.** La decisión existe antes que el operador. `$switch` solo la materializa.

In [ ]:
#@title Pregunta 1 — Pipeline y vista { display-mode: "form" }
pregunta_interactiva_codificada("eyJudW1lcm8iOjEsInRlbWEiOiJQaXBlbGluZSB5IHZpc3RhIiwicHJlZ3VudGEiOiLCv0N1w6FsIGFmaXJtYWNpw7NuIGRlc2NyaWJlIG1lam9yIHVuYSB2aXN0YSBjcmVhZGEgZGVzZGUgdW4gcGlwZWxpbmUgZGUgYWdyZWdhY2nDs24/Iiwib3BjaW9uZXMiOlsiRHVwbGljYSBsb3MgZG9jdW1lbnRvcyBwYXJhIHF1ZSBsYSBjb25zdWx0YSBzZWEgbcOhcyByw6FwaWRhLiIsIkNvbnNlcnZhIHVuYSBkZWZpbmljacOzbiBkZSBzb2xvIGxlY3R1cmEgcXVlIHNlIGV2YWzDumEgZGVzZGUgbG9zIGRhdG9zIGRlIG9yaWdlbi4iLCJFcyBleGFjdGFtZW50ZSBsbyBtaXNtbyBxdWUgZ3VhcmRhciB1biBwaXBlbGluZSBlbiBlbCBlZGl0b3IuIiwiUGVybWl0ZSBlc2NyaWJpciBzb2JyZSBlbCByZXN1bHRhZG8gc2luIG1vZGlmaWNhciBsYSBjb2xlY2Npw7NuIGRlIG9yaWdlbi4iXSwiY29ycmVjdGEiOjEsInJldHJvIjpbIk5vLiBVbmEgdmlzdGEgZXN0w6FuZGFyIG5vIGNyZWEgdW5hIGNvcGlhIGluZGVwZW5kaWVudGUgZGUgbG9zIGRvY3VtZW50b3MuIiwiU8OtLiBMYSB2aXN0YSBjb25zZXJ2YSBsYSBkZWZpbmljacOzbiB5IGV4cG9uZSBzdSByZXN1bHRhZG8gY29tbyB1biBvYmpldG8gY29uc3VsdGFibGUgZGUgc29sbyBsZWN0dXJhLiIsIk5vLiBHdWFyZGFyIGxhIHJlY2V0YSB5IGNyZWFyIHVuYSB2aXN0YSBzb24gb3BlcmFjaW9uZXMgZGlzdGludGFzLiIsIk5vLiBMYSB2aXN0YSBlc3TDoW5kYXIgZXMgZGUgc29sbyBsZWN0dXJhLiJdfQ==")

### MODELO MENTAL VISTA S05 — publicar una transformación sin copiar los datos

```text
entidades_noticias
      │
      │ pipeline
      ▼
menciones_clasificadas
      (VIEW)
```

Piensa en una vista como una **pregunta guardada y consultable**, parecida mentalmente a una `VIEW` de SQL:

| La vista... | ¿Sí o no? |
|---|---|
| crea otros 142 documentos independientes | **No** |
| guarda la definición del pipeline | **Sí** |
| calcula su resultado cuando la consultas | **Sí** |
| se consulta como un objeto propio | **Sí** |
| se edita como una colección normal | **No: es de solo lectura** |

**Ejemplo pequeño.** Si `entidades_noticias` cambia y una entidad pasa de 4 a 5 noticias, la lógica de la vista vuelve a evaluar el pipeline cuando se consulta; no tenemos que mantener manualmente una segunda copia.

**Error frecuente.** Confundir **guardar el pipeline** con **crear la vista**. El primero conserva la receta en el editor; el segundo publica esa receta como objeto consultable.

---
## 3. Tutorial visual 1 — Atlas: construir la vista que sí viaja

**HAZ ESTO AHORA.** Trabaja en Atlas y vuelve a este mismo cuaderno.

No repetimos registro, clúster, carga, filtros de calentamiento ni pipelines que no se consumen después.

Al regresar deben existir:

- pipeline guardado `clasificar-menciones-v1`;
- vista **`menciones_clasificadas`**;
- control `6 alta + 25 media + 111 baja = 142`.

**MÁS ADELANTE.** `resumen-secciones-v1` y `clasificar-noticias-v1` quedan como ampliación para recuperar tiempo de laboratorio.

In [ ]:
#@title Tutorial 1 — Atlas: vista paso a paso { display-mode: "form" }
tutorial(
    "https://jazaineam1.github.io/BigData2026/assets/tutoriales/atlas-s05-pipelines-vistas-v3.html",
    alto=780,
    titulo="Tutorial Atlas — consultas, pipelines y vistas"
)

### Control antes de avanzar

No continúes por intuición. Debes poder responder **sí** a estas tres preguntas:

- [ ] ¿veo `menciones_clasificadas` como una vista?
- [ ] ¿puedo explicar qué hace `$switch` en el pipeline?
- [ ] ¿sé la diferencia entre guardar el pipeline y crear la vista?

Si no, vuelve a la diapositiva correspondiente. El bloque siguiente consume esa vista.

---
## 4. Consumir la vista y comprobar su resultado

Ahora el cuaderno vuelve a ser el lugar donde **se integra** la evidencia.

Si la vista existe en Atlas, la traemos tal como la construiste.  
Si no existe o la conexión falló, se reconstruye el mismo resultado desde el archivo versionado para que nadie pierda el resto de la clase.

**El respaldo permite aprender; no prueba que hayas completado el paso de Atlas.**

In [ ]:
import json, urllib.request
from collections import Counter

vista_real = False

if db is not None:
    nombres = db.list_collection_names()
    if "menciones_clasificadas" in nombres:
        menciones = list(db["menciones_clasificadas"].find({}, {"_id": 0}))
        vista_real = True
    else:
        print("La vista no aparece en Atlas. Activo respaldo con la MISMA regla.")

if not vista_real:
    with urllib.request.urlopen("https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/entidades_en_noticias_2026.json") as r:
        menciones = json.loads(r.read().decode("utf-8"))
    for e in menciones:
        n = e.get("noticias", 0)
        e["nivel_menciones"] = "alta" if n >= 20 else "media" if n >= 5 else "baja"

niveles = Counter(m["nivel_menciones"] for m in menciones)
print("Fuente:", "vista real de Atlas" if vista_real else "respaldo reproducible")
print("Entidades:", len(menciones))
print("Niveles:", dict(niveles))

assert len(menciones) == 142
assert dict(niveles) == {"baja": 111, "media": 25, "alta": 6}

### Cómo leer esta salida

**Cómo se lee.** De 142 entidades asociadas a las noticias, 6 quedan en nivel alto, 25 en medio y 111 en bajo según una regla basada en número de noticias.

**Qué nos dice.** Ya tenemos una variable explicable que resume intensidad de mención **por entidad**.

**Qué NO permite concluir todavía.** No sabemos si una noticia habla de un contrato específico. Nos falta comparar referencias de proceso.

**Error frecuente.** Leer “nivel alto” como “riesgo alto”. Aquí solo significa **más menciones según el umbral que definimos**.

---
## 5. Atlas + SECOP: convertir una vista en una bandeja

La vista por sí sola no responde la pregunta de Laura. La cruzamos con una muestra de 1.000 procesos SECOP mediante una regla explícita:

```text
1. entidad aparece en prensa
2. modalidad contiene "directa"
3. respuestas al procedimiento = 0
4. ordenar por precio_base DESC
```

No es un modelo predictivo. Es una regla transparente que cualquiera puede discutir.

### MICROEJEMPLO MERGE S05 — muchos procesos pueden heredar el contexto de una entidad

Antes del cruce real, mira este caso diminuto.

**Contexto de prensa**

| entidad | noticias | nivel |
|---|---:|---|
| A | 24 | alta |
| B | 6 | media |

**Procesos SECOP**

| proceso | entidad | valor |
|---|---|---:|
| P1 | A | 100 |
| P2 | A | 50 |
| P3 | C | 80 |

Al unir por `entidad`, P1 y P2 reciben el mismo contexto de A. Eso es un patrón **many-to-one**: muchos procesos pueden apuntar a una sola fila de contexto por entidad.

```text
P1 ─┐
    ├─ Entidad A → 24 noticias → alta
P2 ─┘
```

Por eso usamos `validate="many_to_one"`: pandas comprueba que la tabla de contexto no tenga dos filas distintas para la misma entidad. Si esa suposición se rompe, preferimos un error explícito a duplicar procesos silenciosamente.

**PARA LLEVAR.** `noticias_entidad` y `nivel_menciones` **viajan como contexto**. No deciden por sí solos quién entra a la bandeja.

### FUNDAMENTO REGLA S05 — una heurística de trabajo, no un detector de fraude

La bandeja usa una regla explícita porque Laura necesita una cola de revisión defendible. Cada condición cumple un papel **operacional**:

| Condición | Para qué la usamos aquí | Lo que **NO** significa |
|---|---|---|
| entidad aparece en prensa | agrega contexto externo para decidir dónde mirar | que la entidad cometió una irregularidad |
| modalidad contiene `directa` | acota el ejercicio a una modalidad concreta | que contratación directa = corrupción |
| respuestas = `0` | describe una característica observada del proceso | que cero respuestas = anomalía |
| `precio_base DESC` | ordena por exposición económica potencial | que mayor valor = mayor probabilidad de irregularidad |

**Ejemplo.** Si dos procesos cumplen la misma regla y uno vale $1.000 millones y otro $20 millones, ordenar el primero antes solo expresa una decisión de **impacto potencial**. No hemos estimado la probabilidad de que exista un problema.

```text
probabilidad de irregularidad  ≠  impacto económico potencial
```

**OJO.** En contratación directa puede existir un proceso sin competencia de múltiples ofertas. Por eso `directa + 0 respuestas` se trata aquí como una **heurística pedagógica versionada**, no como dos “señales de corrupción”.

**PARA LLEVAR.** La regla organiza revisión humana. No produce un veredicto.

### MICROEJEMPLO NAN S05 — desconocido no es lo mismo que cero

| valor original | `pd.to_numeric(..., errors="coerce")` | `.eq(0)` |
|---|---:|---|
| `"0"` | `0` | `True` |
| `"2"` | `2` | `False` |
| vacío | `NaN` | `False` |
| `"No definido"` | `NaN` | `False` |

Si hiciéramos `fillna(0)`, transformaríamos **“no conozco el dato”** en **“observé exactamente cero”**. Eso cambiaría la evidencia y podría meter filas a la bandeja por una imputación que nunca justificamos.

In [ ]:
import pandas as pd

secop = pd.read_csv(
    "https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv",
    low_memory=False,
)
print("Procesos SECOP:", len(secop))
assert len(secop) == 1000

contexto_menciones = pd.DataFrame(menciones)[["entidad", "noticias", "nivel_menciones"]].copy()
assert contexto_menciones["entidad"].is_unique
contexto_menciones = contexto_menciones.rename(columns={"noticias": "noticias_entidad"})

entidades_en_prensa = set(contexto_menciones["entidad"])
paso1 = secop[secop["entidad"].isin(entidades_en_prensa)]
paso2 = paso1[
    paso1["modalidad_de_contratacion"].str.contains("directa", case=False, na=False)
]
respuestas = pd.to_numeric(paso2["respuestas_al_procedimiento"], errors="coerce")
paso3 = paso2[respuestas.eq(0)]

candidatos = (
    paso3
    .merge(contexto_menciones, on="entidad", how="left", validate="many_to_one")
    .sort_values(["precio_base", "id_del_proceso"], ascending=[False, True])
    .reset_index(drop=True)
)

print("1) total                       :", len(secop))
print("2) entidad coincide con prensa :", len(paso1))
print("3) + modalidad directa         :", len(paso2))
print("4) + cero respuestas           :", len(paso3))
print("Candidatos finales             :", len(candidatos))
print("Contexto de menciones en los 77:", candidatos["nivel_menciones"].value_counts().to_dict())

assert len(paso1) == 163
assert len(candidatos) == 77
assert candidatos["nivel_menciones"].notna().all()

### INTERPRETACIÓN EMBUDO S05

**Cómo se lee.** Partimos de 1.000 procesos, 163 pertenecen a entidades presentes en noticias y 77 sobreviven a modalidad directa + cero respuestas. Cada candidato conserva `noticias_entidad` y `nivel_menciones` de la vista.

**Qué nos dice.** La vista aporta contexto explicable a cada fila que Laura recibirá.

**Qué NO permite concluir todavía.** `nivel_menciones` no es criterio de selección ni probabilidad de riesgo; todavía no sabemos si una noticia menciona el contrato particular.

**Error frecuente.** Creer que “alta” empujó un proceso dentro de los 77. El nivel viaja como contexto, no como filtro.

### El detalle estadístico que importa: faltante no es cero

La celda anterior usa:

```python
pd.to_numeric(..., errors="coerce").eq(0)
```

y **no** usa:

```python
fillna(0).eq(0)
```

¿Por qué? Porque “no conocemos el número de respuestas” y “hubo exactamente cero respuestas” son eventos distintos.
Convertir el faltante a cero fabricaría evidencia que no existe.

In [ ]:
primero = candidatos.iloc[0]

print("Primer candidato")
print("ID                :", primero["id_del_proceso"])
print("Entidad           :", primero["entidad"])
print("Valor             : $", f'{primero["precio_base"]:,.0f}')
print("Noticias entidad  :", int(primero["noticias_entidad"]))
print("Nivel de menciones:", primero["nivel_menciones"])

assert primero["id_del_proceso"] == "CO1.REQ.5407319"
assert primero["entidad"] == "MINISTERIO DEL DEPORTE"
assert int(primero["precio_base"]) == 168750000

---
## 6. Antes de afirmar: formula una hipótesis que pueda fallar

La bandeja ya existe. Ahora hacemos algo distinto de filtrar: **ponemos a prueba una afirmación sobre la prensa**.

La pregunta no es “¿la prensa sirve o no sirve?”. Esa pregunta es demasiado vaga. La hacemos observable:

> **H1: al menos una de las 77 referencias SECOP exactas aparece literalmente en los títulos o subtítulos examinados.**

¿Por qué esta formulación es mejor?

```text
afirmación vaga
"la prensa habla de estos contratos"
        ↓
hipótesis observable
"aparece al menos un ID exacto"
        ↓
regla de comprobación
buscar los 77 IDs en título + subtítulo
```

Si aparece al menos uno, H1 sobrevive a esta prueba literal. Si aparecen cero, H1 queda refutada **bajo esta operacionalización**.

**Importante:** esto no es todavía un test estadístico inferencial. Es un contraste empírico de una hipótesis de trabajo sobre este corpus.

**PARA LLEVAR.** La prensa no entra al evaluador para dictar culpabilidad. Entra para aportar contexto, permitir formular hipótesis y ayudarnos a decidir cuál debe ser la siguiente evidencia.

In [ ]:
with urllib.request.urlopen("https://raw.githubusercontent.com/jazaineam1/BigData2026/main/Datos/noticias_contratacion_2026.json") as r:
    noticias = json.loads(r.read().decode("utf-8"))

texto_prensa = " ".join(
    f'{n.get("titulo") or ""} {n.get("subtitulo") or ""}'
    for n in noticias
).casefold()

def citado_literalmente(fila):
    referencia = str(fila.get("referencia_del_proceso") or "").strip()
    return bool(referencia) and len(referencia) >= 6 and referencia.casefold() in texto_prensa

candidatos = candidatos.copy()
candidatos["referencia_citada_en_prensa"] = candidatos.apply(citado_literalmente, axis=1)

con_referencia = int(candidatos["referencia_citada_en_prensa"].sum())
print("Referencias citadas literalmente:", con_referencia, "de", len(candidatos))

assert con_referencia == 0

### Cómo leer esta salida

**Cómo se lee.** Ninguno de los 77 candidatos tiene su referencia de proceso citada literalmente en los títulos o subtítulos de las noticias usadas.

**Qué nos dice.** La regla puede ayudar a decidir **dónde mirar primero**.

**Qué NO permite concluir todavía.** No demuestra que esos 77 procesos hayan sido cuestionados por la prensa, ni fraude, irregularidad o incumplimiento.

**Error frecuente.** Decir “la prensa señaló estos contratos”. No: la evidencia periodística que usamos está asociada a la **entidad**, no al contrato específico.

In [ ]:
#@title Pregunta 2 — Qué podemos afirmar { display-mode: "form" }
pregunta_interactiva_codificada("eyJudW1lcm8iOjIsInRlbWEiOiJMw61taXRlIGFuYWzDrXRpY28iLCJwcmVndW50YSI6IsK/Q3XDoWwgYWZpcm1hY2nDs24gcHVlZGUgc29zdGVuZXIgTGF1cmEgZGVzcHXDqXMgZGVsIDAgZGUgNzc/Iiwib3BjaW9uZXMiOlsiRW5jb250cmFtb3MgNzcgY29udHJhdG9zIGlycmVndWxhcmVzLiIsIkxhIHByZW5zYSBpZGVudGlmaWPDsyBkaXJlY3RhbWVudGUgbG9zIDc3IGNvbnRyYXRvcy4iLCJQcmlvcml6YW1vcyA3NyBwcm9jZXNvcyBwb3Igc2XDsWFsZXMgZXhwbGljYWJsZXM7IGxhIGV2aWRlbmNpYSBkZSBwcmVuc2EgZXMgcG9yIGVudGlkYWQgeSByZXF1aWVyZSByZXZpc2nDs24gaHVtYW5hLiIsIkNvbW8gMCBkZSA3NyBlc3TDoSBjaXRhZG8sIGxhIHJlZ2xhIG5vIHNpcnZlIHBhcmEgbmFkYS4iXSwiY29ycmVjdGEiOjIsInJldHJvIjpbIk5vLiBVbmEgcmVnbGEgZGUgcHJpb3JpZGFkIG5vIHBydWViYSBpcnJlZ3VsYXJpZGFkLiIsIk5vLiBFbCBjb250cm9sIDAvNzcgbXVlc3RyYSBqdXN0YW1lbnRlIGxvIGNvbnRyYXJpby4iLCJDb3JyZWN0by4gTGEgc2FsaWRhIHNpcnZlIHBhcmEgcHJpb3JpemFyLCB5IHN1IGzDrW1pdGUgdmlhamEgY29uIGVsbGEuIiwiTm8uIEVsIGzDrW1pdGUgcmVkdWNlIGxhIGZ1ZXJ6YSBkZSBsYSBhZmlybWFjacOzbiwgbm8gZWxpbWluYSBlbCB2YWxvciBvcGVyYXRpdm8gZGUgcHJpb3JpemFyLiJdfQ==")

---
# 7. Cassandra aparece por una necesidad, no por el cronograma

Laura ya tiene una bandeja. Imagina ahora este patrón:

> “Para **este corte** y **este departamento**, dame los **5 procesos de mayor valor** que debo revisar primero.”

Hoy pregunta una persona. Mañana pueden ser cien analistas consultando lo mismo una y otra vez.

Con 77 filas **no necesitas Cassandra**: pandas responde sobrado.  
La razón de estudiarlo es aprender un patrón que sigue funcionando cuando la bandeja crece y la misma consulta se repite a escala.

## Query-first design

En Cassandra no empezamos preguntando “¿qué entidades existen?”. Empezamos por:

> **¿Qué consulta debo servir de forma barata y repetitiva?**

Luego diseñamos la tabla alrededor de ella.

### MICROEJEMPLO PAPEL CASSANDRA S05 — resuelve la consulta antes de conocer Cassandra

Laura tiene estas cinco filas:

| corte | departamento | proceso | valor |
|---|---|---|---:|
| 03-sep | Bogotá | P1 | 50 |
| 03-sep | Bogotá | P2 | 180 |
| 03-sep | Bogotá | P3 | 90 |
| 03-sep | Antioquia | P4 | 300 |
| 02-sep | Bogotá | P5 | 500 |

Pregunta: **para el corte 03-sep y Bogotá, ¿qué tres procesos debe abrir primero si ordenamos por valor?**

```text
1. P2 → 180
2. P3 →  90
3. P1 →  50
```

Antes de pensar en sintaxis ya sabemos dos cosas:

1. necesitamos **localizar** el grupo `03-sep + Bogotá`;
2. dentro de ese grupo necesitamos **ordenar** de mayor a menor valor.

Cassandra aparecerá después como una forma de materializar exactamente esas dos decisiones.

### ESCENA OPERACIONAL CASSANDRA S05 — primero la consulta de negocio

Con **77 filas**, pandas ya responde. Cassandra aparece para practicar el diseño de un servicio cuando esta lectura se vuelve estable y repetitiva.

Imagina que Compras Claras ya tiene una interfaz para varios revisores:

```text
GET /prioridades?corte=2026-09-03&departamento=Bogota&limit=5
```

La pregunta detrás de esa llamada es:

> **Para este corte y este departamento, dame primero los procesos de mayor valor.**

Ahora cada término tiene sentido de negocio:

| Dato | Qué representa |
|---|---|
| `corte` | la versión/fecha de la bandeja |
| `departamento` | la cola territorial que atiende un revisor |
| `valor_base DESC` | el orden de exposición económica dentro de esa cola |
| `LIMIT 5` | qué abre primero el revisor |

**PARA LLEVAR.** No usamos Cassandra porque 77 filas “sean Big Data”. La usamos para aprender **query-first design**: una tabla nace de una consulta que sabemos que el servicio debe atender repetidamente.

### MICROEJEMPLO PARTICIONES S05 — piensa en cajones antes de mirar la `PRIMARY KEY`

```text
ARCHIVADOR DE PRIORIDADES

Cajón: 2026-09-03 + Bogotá
├── $180 M → P8
├── $120 M → P3
└──  $40 M → P5

Cajón: 2026-09-03 + Antioquia
├── $250 M → P9
└──  $70 M → P2
```

- **Partition key `(corte, departamento)`** = la etiqueta del cajón que permite localizar el grupo.
- **Clustering `valor_base DESC, id_proceso ASC`** = cómo quedan ordenadas las filas **dentro** del cajón.

Por eso, en esta sesión `PRIMARY KEY` significa más que “un identificador único”:

```text
PRIMARY KEY
├── partition key      → dónde viven los datos
└── clustering columns → cómo se organizan dentro de la partición
```

**Error frecuente.** Diseñar la clave mirando qué columnas parecen importantes. En Cassandra la pregunta profesional va primero.

### TRADUCCIÓN PK EN TRES PASOS S05 — no memorices los paréntesis

```text
PASO 1 · localizar el cajón
(corte, departamento)

PASO 2 · ordenar dentro del cajón
valor_base DESC, id_proceso ASC

PASO 3 · traducirlo a CQL
PRIMARY KEY ((corte, departamento), valor_base, id_proceso)
```

Se lee así:

```text
((corte, departamento))  → PARTITION KEY → dónde buscar
valor_base, id_proceso   → CLUSTERING    → cómo ordenar dentro
```

**Objetivo de aprendizaje.** No debes reconstruir los paréntesis de memoria. Debes mirar una clave y poder decir **qué localiza la partición y qué organiza sus filas**.

### EJERCICIO S05-PK — Elige antes de mirar la respuesta

La consulta profesional es:

> **Para un corte y un departamento, devolver primero los procesos de mayor valor.**

¿Cuál diseño permite localizar directamente el grupo que Laura conoce al consultar?

- **A.** `PRIMARY KEY (id_proceso)`
- **B.** `PRIMARY KEY ((corte, departamento), valor_base, id_proceso)`
- **C.** `PRIMARY KEY ((entidad), id_proceso)`

No es una pregunta calificable. La decisión importa más que acertar de memoria: elige una opción y explica en una frase por qué descartas otra.

In [ ]:
eleccion_pk = input("Tu elección (A, B o C): ").strip().upper()
razon_descartada = input("Descarta una alternativa en una frase: ").strip()

if eleccion_pk == "B":
    print("Bien: la consulta conoce corte + departamento y puede localizar esa partición.")
elif eleccion_pk in {"A", "C"}:
    print("Revisa el patrón de acceso: Laura conoce corte + departamento, no un id ni necesariamente una entidad.")
else:
    print("Escribe A, B o C. Luego vuelve a leer la pregunta profesional.")

print("Alternativa descartada:", razon_descartada or "PENDIENTE")

### La llave se lee en dos partes

```sql
PRIMARY KEY (
    (corte, departamento),
    valor_base,
    id_proceso
)
```

| Parte | Papel | Pregunta |
|---|---|---|
| `(corte, departamento)` | clave de partición | ¿a qué partición debo ir? |
| `valor_base` | clustering | ¿cómo quedan ordenadas las filas dentro de la partición? |
| `id_proceso` | clustering/desempate | ¿cómo identifico una fila de forma estable si dos valores empatan? |

Con:

```sql
CLUSTERING ORDER BY (valor_base DESC, id_proceso ASC)
```

la partición ya queda preparada para responder:

```sql
WHERE corte = ? AND departamento = ?
LIMIT 5
```

sin tener que ordenar todo después.

In [ ]:
#@title Pregunta 3 — Clave de partición { display-mode: "form" }
pregunta_interactiva_codificada("eyJudW1lcm8iOjMsInRlbWEiOiJRdWVyeS1maXJzdCBkZXNpZ24iLCJwcmVndW50YSI6IsK/UG9yIHF1w6kgbGEgdGFibGEgdXNhIChjb3J0ZSwgZGVwYXJ0YW1lbnRvKSBjb21vIGNsYXZlIGRlIHBhcnRpY2nDs24/Iiwib3BjaW9uZXMiOlsiUG9ycXVlIHNvbiBsYXMgY29sdW1uYXMgbcOhcyBpbXBvcnRhbnRlcyBkZWwgbmVnb2Npby4iLCJQb3JxdWUgbGEgY29uc3VsdGEgcmVwZXRpdGl2YSBjb25vY2UgZXNvcyBkb3MgdmFsb3JlcyB5IHB1ZWRlIGxvY2FsaXphciB1bmEgcGFydGljacOzbiBjb25jcmV0YS4iLCJQb3JxdWUgQ2Fzc2FuZHJhIGV4aWdlIGV4YWN0YW1lbnRlIGRvcyBjb2x1bW5hcyBlbiB0b2RhIGNsYXZlIGRlIHBhcnRpY2nDs24uIiwiUG9ycXVlIGFzw60gc2UgcHVlZGVuIGhhY2VyIGpvaW5zIGNvbiBNb25nb0RCLiJdLCJjb3JyZWN0YSI6MSwicmV0cm8iOlsiTm8gYmFzdGEgY29uIHF1ZSB1biBjYW1wbyBzZWEgaW1wb3J0YW50ZTogbGEgY2xhdmUgbmFjZSBkZWwgcGF0csOzbiBkZSBhY2Nlc28uIiwiQ29ycmVjdG8uIExhIHBhcnRpY2nDs24gc2UgZGlzZcOxYSBkZXNkZSBsYSBjb25zdWx0YSBxdWUgcXVlcmVtb3Mgc2VydmlyLiIsIk5vLiBVbmEgY2xhdmUgZGUgcGFydGljacOzbiBwdWVkZSB0ZW5lciB1bmEgbyB2YXJpYXMgY29sdW1uYXMuIiwiTm8uIENhc3NhbmRyYSBubyBpbnRyb2R1Y2Ugam9pbnMgY29uIE1vbmdvREI7IGxhIGJhbmRlamEgc2UgcHJlcGFyYSBhbnRlcy4iXX0=")

## ¿Para qué datos encaja este patrón?

| Encaja bien | No es la primera opción |
|---|---|
| pocas consultas conocidas de antemano | exploración donde la pregunta cambia cada cinco minutos |
| muchas escrituras y lecturas repetitivas | joins frecuentes entre entidades normalizadas |
| distribución horizontal y alta disponibilidad | transacciones relacionales complejas |
| datos desnormalizados para servir una pregunta | “guardar una vez y preguntar cualquier cosa después” |

**PARA LLEVAR.** Cassandra no es “MongoDB pero más rápido”. Hace otro compromiso:
te obliga a preparar el almacenamiento para unas consultas concretas.

### CHULETA CQL S05 — cinco comandos, una sola historia

| Comando | Para qué sirve hoy | Qué debes observar | Error frecuente |
|---|---|---|---|
| `USE compras_claras;` | trabajar en el keyspace creado en Astra | la consola cambia de contexto | intentar `CREATE KEYSPACE` en Astra |
| `CREATE TABLE` | definir la tabla para la consulta objetivo | la `PRIMARY KEY` nace del patrón de acceso | diseñar por columnas “importantes” |
| `INSERT INTO` | escribir una fila | columnas y valores corresponden | pensar que es una carga analítica masiva |
| `SELECT ... WHERE` | leer una partición | `WHERE` conoce `corte + departamento` | filtrar cualquier columna porque existe |
| `UPDATE` / `DELETE` | cambiar o borrar una fila identificada | se usa la clave necesaria para identificarla | olvidar parte de la clave |

**PARA LLEVAR.** CQL se parece visualmente a SQL; Cassandra no hereda por eso el mismo modelo de consultas ad hoc.

### CONTRATO DE RESULTADO S05 — fija primero qué debería devolver Cassandra

Elige un departamento mediante número. pandas calcula el top 5 esperado; después Cassandra debe devolver los mismos IDs y en el mismo orden.

In [ ]:
# ✏️ MODIFICA · aquí tu elección sí cambia la evidencia individual.
conteo_departamentos = candidatos["departamento_entidad"].dropna().astype(str).value_counts()
opciones_departamento = conteo_departamentos.index.tolist()

print("Elige un departamento para tu evidencia individual:")
for i, d in enumerate(opciones_departamento, start=1):
    print(f"{i:>2}. {d} ({conteo_departamentos[d]} candidatos)")

seleccion = int(input("Número de departamento: ").strip())
if not 1 <= seleccion <= len(opciones_departamento):
    raise ValueError("El número no corresponde a la lista mostrada.")

departamento_elegido = opciones_departamento[seleccion - 1]
top5_esperado_pd = (
    candidatos[candidatos["departamento_entidad"].astype(str) == departamento_elegido]
    .sort_values(["precio_base", "id_del_proceso"], ascending=[False, True])
    .head(5)
)
ids_esperados_pd = top5_esperado_pd["id_del_proceso"].astype(str).tolist()

print("Departamento elegido:", departamento_elegido)
print("IDs esperados por pandas:", ids_esperados_pd)

# RECUPERACIÓN S05
if "candidatos" not in globals():
    import json, urllib.request
    import pandas as pd
    from collections import Counter

    RAW = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main"

    with urllib.request.urlopen(f"{RAW}/Datos/entidades_en_noticias_2026.json") as r:
        menciones = json.loads(r.read().decode("utf-8"))

    for e in menciones:
        n = e.get("noticias", 0)
        e["nivel_menciones"] = "alta" if n >= 20 else "media" if n >= 5 else "baja"

    niveles = Counter(m["nivel_menciones"] for m in menciones)
    contexto_menciones = (
        pd.DataFrame(menciones)[["entidad", "noticias", "nivel_menciones"]]
        .rename(columns={"noticias": "noticias_entidad"})
    )
    assert contexto_menciones["entidad"].is_unique
    vista_real = False

    secop = pd.read_csv(
        f"{RAW}/Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv",
        low_memory=False,
    )
    entidades_en_prensa = set(contexto_menciones["entidad"])
    paso1 = secop[secop["entidad"].isin(entidades_en_prensa)]
    paso2 = paso1[
        paso1["modalidad_de_contratacion"].str.contains("directa", case=False, na=False)
    ]
    respuestas = pd.to_numeric(paso2["respuestas_al_procedimiento"], errors="coerce")
    paso3 = paso2[respuestas.eq(0)]
    candidatos = (
        paso3
        .merge(contexto_menciones, on="entidad", how="left", validate="many_to_one")
        .sort_values(["precio_base", "id_del_proceso"], ascending=[False, True])
        .reset_index(drop=True)
    )

    with urllib.request.urlopen(f"{RAW}/Datos/noticias_contratacion_2026.json") as r:
        noticias = json.loads(r.read().decode("utf-8"))
    texto_prensa = " ".join(
        f'{n.get("titulo") or ""} {n.get("subtitulo") or ""}' for n in noticias
    ).casefold()
    referencias = candidatos["referencia_del_proceso"].fillna("").astype(str).str.strip()
    con_referencia = sum(
        bool(x) and len(x) >= 6 and x.casefold() in texto_prensa
        for x in referencias
    )

    assert len(menciones) == 142
    assert dict(niveles) == {"baja": 111, "media": 25, "alta": 6}
    assert len(secop) == 1000
    assert len(paso1) == 163
    assert len(candidatos) == 77
    assert candidatos["noticias_entidad"].notna().all()
    assert candidatos["nivel_menciones"].notna().all()
    assert con_referencia == 0
    print("Runtime recuperado: 142 entidades · 1.000 → 163 → 77 · 0/77 · contexto Atlas restaurado.")
else:
    print("La bandeja sigue en memoria:", len(candidatos), "candidatos. No fue necesario reconstruirla.")

In [ ]:
#@title Recuperar la bandeja si Colab reinició { display-mode: "form" }
# RECUPERACIÓN S05
if "candidatos" not in globals():
    import json, urllib.request
    import pandas as pd
    from collections import Counter

    RAW = "https://raw.githubusercontent.com/jazaineam1/BigData2026/main"

    with urllib.request.urlopen(f"{RAW}/Datos/entidades_en_noticias_2026.json") as r:
        menciones = json.loads(r.read().decode("utf-8"))
    for e in menciones:
        n = e.get("noticias", 0)
        e["nivel_menciones"] = "alta" if n >= 20 else "media" if n >= 5 else "baja"
    niveles = Counter(m["nivel_menciones"] for m in menciones)
    vista_real = False

    secop = pd.read_csv(
        f"{RAW}/Cuadernos/datos/secop_chunks/prueba_chunk_0000000.csv",
        low_memory=False,
    )
    entidades_en_prensa = {m["entidad"] for m in menciones}
    paso1 = secop[secop["entidad"].isin(entidades_en_prensa)]
    paso2 = paso1[
        paso1["modalidad_de_contratacion"].str.contains("directa", case=False, na=False)
    ]
    respuestas = pd.to_numeric(paso2["respuestas_al_procedimiento"], errors="coerce")
    candidatos = paso2[respuestas.eq(0)].sort_values(
        ["precio_base", "id_del_proceso"], ascending=[False, True]
    ).reset_index(drop=True)

    with urllib.request.urlopen(f"{RAW}/Datos/noticias_contratacion_2026.json") as r:
        noticias = json.loads(r.read().decode("utf-8"))
    texto_prensa = " ".join(
        f'{n.get("titulo") or ""} {n.get("subtitulo") or ""}' for n in noticias
    ).casefold()
    referencias = candidatos["referencia_del_proceso"].fillna("").astype(str).str.strip()
    con_referencia = sum(bool(x) and len(x) >= 6 and x.casefold() in texto_prensa for x in referencias)

    assert len(menciones) == 142
    assert dict(niveles) == {"baja": 111, "media": 25, "alta": 6}
    assert len(secop) == 1000
    assert len(paso1) == 163
    assert len(candidatos) == 77
    assert con_referencia == 0
    print("Runtime recuperado: 142 entidades · 1.000 → 163 → 77 · 0/77.")
else:
    print("La bandeja sigue en memoria:", len(candidatos), "candidatos. No fue necesario reconstruirla.")

### MODELO MENTAL ASTRA S05 — cuatro nombres antes de hacer clic

| Nombre | Qué significa hoy |
|---|---|
| **Astra DB** | servicio administrado donde usaremos Cassandra sin instalar un servidor en Windows |
| **Cassandra** | motor/modelo wide-column que estamos estudiando |
| **CQL** | lenguaje para definir y consultar tablas Cassandra |
| **keyspace** | contenedor lógico donde agrupamos las tablas de `compras_claras` |

```text
Astra database
└── keyspace compras_claras
    └── prioridades_por_corte_departamento
```

Elegimos **Serverless (non-vector)** porque hoy trabajamos una tabla Cassandra/CQL por claves. No estamos haciendo embeddings ni búsqueda vectorial.

**PARA LLEVAR.** No necesitas instalar Cassandra localmente ni entender estrategias de replicación para completar esta práctica; Astra administra esa infraestructura.

---
## 8. PRODUCTO 3 · Tramo A — implementar la pregunta en Astra/CQL

**▶️ EJECUTA.** Ahora sí llevamos el diseño a la interfaz. No estás aprendiendo “otra historia”: estás implementando la consulta que ya resolviste en papel.

**🧠 ENTIENDE:** `partition → clustering → consulta soportada / consulta no soportada`.  
**▶️ EJECUTA:** crear base, esperar `Active`, abrir CQL Console, crear tabla y probar CQL.  
**✏️ MODIFICA:** valores de consulta cuando el tutorial te lo indique.

## 8. Tutorial visual 2 — Astra DB y CQL, paso a paso

- el **cuaderno** explica modelo y decisión;
- el **tutorial HTML** reproduce la ruta de interfaz y los nombres de los controles;
- tú ejecutas cada paso en Astra;
- vuelves con una tabla real.

**Transparencia visual.** Donde no existe una captura autenticada de una cuenta del curso usamos una **representación de interfaz claramente rotulada**, no una fotografía fingida.

Ruta del grupo: Astra DB Serverless **non-vector** + CQL Console. No instalamos Cassandra en Windows ni en Colab.

Ruta contrastada con documentación oficial vigente el **31 de agosto de 2026**. `Connection details` es el nombre vigente para bases non-vector.

In [ ]:
#@title Tutorial 2 — Astra/Cassandra: guía visual { display-mode: "form" }
tutorial(
    "https://jazaineam1.github.io/BigData2026/assets/tutoriales/astra-cassandra-paso-a-paso-v3.html",
    alto=790,
    titulo="Tutorial Astra — de cero a CQL y conexión con Python"
)

### Control antes de Python

Debes tener:

- [ ] una base Astra DB Serverless **non-vector** activa;
- [ ] keyspace `compras_claras`;
- [ ] tabla `prioridades_por_corte_departamento`;
- [ ] al menos una inserción hecha en CQL Console;
- [ ] una consulta por `corte + departamento` que funciona;
- [ ] tu **Secure Connect Bundle** descargado;
- [ ] un token de aplicación guardado fuera del cuaderno.

**Nunca pegues el token en una celda Markdown ni lo subas a GitHub.**

### CHECKPOINT CQL S05 — ya entendiste Cassandra antes de conectar Python

Antes de abrir SCB/token, comprueba que puedes contestar:

1. ¿Qué consulta repetitiva estamos sirviendo?  
2. ¿Cuál es el “cajón” de esa consulta?  
3. ¿Cómo se ordenan sus filas?  
4. ¿Por qué consultar solo por `entidad` requiere otro diseño?

Si puedes responder esas cuatro preguntas, **la comprensión central de Cassandra ya está**. El siguiente tramo conserva Python porque el PDA pide CRUD e integración, pero no introduce otro modelo de datos: automatiza **la misma tabla**.

---
## 9. PRODUCTO 3 · Tramo B — la misma tabla, ahora desde Python

El PDA pide CRUD con Python. **No es un cuarto producto ni otro motor.** Una aplicación usa la misma tabla Cassandra que acabas de diseñar y probar en CQL Console.

### Qué debes hacer con esta parte

- ▶️ **EJECUTA** la instalación, carga del SCB y creación de `Cluster/Session`.
- 🧠 **ENTIENDE** que SCB = ruta segura, token = autenticación, `Session` = canal para ejecutar CQL.
- ✏️ **MODIFICA** el departamento/consulta en los puntos señalados y compara el resultado.
- 🧠 **ENTIENDE** CRUD como cuatro operaciones sobre la misma tabla; no memorices la ceremonia de conexión.

```text
CQL Console                 Python
-----------                 ------
CREATE / INSERT / SELECT →  session.execute(...)
misma tabla              →  misma pregunta
misma partition key      →  mismo resultado esperado
```

**PARA LLEVAR.** Python automatiza lo que ya comprendiste en CQL; no cambia el modelo Cassandra.

### MODELO CONEXIÓN PYTHON S05 — no memorices objetos: asigna una función a cada uno

```text
SCB      → ¿cómo llega el driver de forma segura a ESTA base?
TOKEN    → ¿con qué credencial me autentico?
Cluster  → cliente Python configurado
Session  → canal con el que ejecuto CQL
prepare  → plantilla CQL con espacios `?` para valores
execute  → envía la plantilla + los valores reales
```

**Ejemplo de `prepare()`:**

```python
consulta = session.prepare("""
SELECT * FROM prioridades_por_corte_departamento
WHERE corte = ? AND departamento = ?
LIMIT 5
""")

session.execute(consulta, (CORTE_CLASE, departamento_elegido))
```

Se lee así:

```text
corte = ?         ← 2026-09-03
departamento = ?  ← Bogotá
```

Los `?` no son valores desconocidos del dataset: son **lugares reservados** que se completan al ejecutar.

**Error frecuente.** Pensar que `Cluster(...)` crea otro clúster en la nube. Aquí solo crea el objeto cliente del driver Python.

### MINI FICHA DRIVER S05

| Objeto | Para qué sirve | Qué recibe | Qué deja |
|---|---|---|---|
| `Cluster(...)` | configura la conexión | SCB + autenticación | cliente del driver |
| `cluster.connect()` | abre la sesión | `Cluster` | `Session` |
| `session.prepare()` | prepara CQL parametrizado | sentencia con `?` | consulta reutilizable |
| `session.execute()` | envía CQL | sentencia + valores | filas o escritura |

**Error frecuente.** Aquí `Cluster` es un objeto del driver Python; no significa crear otro recurso en Astra.

In [ ]:
# ▶️ EJECUTA · soporte de conexión; no memorices esta celda.
!pip install -q cassandra-driver

import importlib.metadata
print("cassandra-driver:", importlib.metadata.version("cassandra-driver"))

### DIAGNÓSTICO ASTRA S05 — si algo falla, identifica primero el síntoma

| Ves | Qué suele significar | Qué haces |
|---|---|---|
| base no está `Active` | aún está provisionando | espera/recarga; no crees otra |
| no aparece `token@cqlsh>` | CQL Console aún no conectó | espera o vuelve a abrir la consola |
| `Unauthorized` | token o permisos | genera/verifica el application token |
| SCB no conecta | bundle de otra base/región o ZIP alterado | descarga de nuevo desde **esta** base |
| consulta por `entidad` falla | **no es instalación** | revisa la clave de partición |

Antes del código recuerda cuatro objetos: **SCB = conexión**, **token = autenticación**, `"token"` = usuario literal del driver, **Cluster/Session = cliente Python**.

In [ ]:
# ▶️ EJECUTA · infraestructura guiada SCB/token; no memorices objetos.
from getpass import getpass
from datetime import date
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider

try:
    from google.colab import files
    print("Sube UN solo Secure Connect Bundle (.zip) de ESTA base:")
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError("Debes subir exactamente un archivo SCB .zip.")
    scb = next(iter(subidos))
except ImportError:
    scb = input("Ruta al Secure Connect Bundle (.zip): ").strip()

if not scb.lower().endswith(".zip"):
    raise ValueError("El Secure Connect Bundle debe conservarse como .zip.")

token_astra = getpass("Application token de Astra (no se muestra): ").strip()
if not token_astra:
    raise ValueError("El token está vacío.")

cluster = Cluster(
    cloud={"secure_connect_bundle": scb},
    auth_provider=PlainTextAuthProvider("token", token_astra),
)
session = cluster.connect()
row = session.execute("SELECT release_version FROM system.local").one()
print("Conectado. Cassandra:", row[0] if row else "versión no disponible")

In [ ]:
cql_tabla = '''
CREATE TABLE IF NOT EXISTS compras_claras.prioridades_por_corte_departamento (
    corte date,
    departamento text,
    valor_base bigint,
    id_proceso text,
    entidad text,
    noticias_entidad int,
    nivel_menciones text,
    estado_revision text,
    url_secop text,
    criterio text,
    PRIMARY KEY ((corte, departamento), valor_base, id_proceso)
) WITH CLUSTERING ORDER BY (valor_base DESC, id_proceso ASC);
'''
session.execute(cql_tabla)
print("Tabla lista.")

### UPSERT S05 — por qué repetir la carga no crea una segunda fila con la misma clave

En Cassandra, un `INSERT` con la misma `PRIMARY KEY` tiene comportamiento de **upsert**: si la fila no existe, la crea; si ya existe, escribe/actualiza los valores de esa misma fila.

```text
1.ª ejecución: P1 no existe  → crea P1
2.ª ejecución: misma PK P1   → actualiza P1, no crea un duplicado P1
```

Esto ayuda a que la carga del laboratorio sea repetible. **No significa** que 77 escrituras síncronas sean la estrategia recomendada para cargas masivas; aquí priorizamos claridad y trazabilidad.

In [ ]:
def texto_seguro(valor, defecto="No definido"):
    if pd.isna(valor):
        return defecto
    texto = str(valor).strip()
    return texto if texto else defecto

insertar = session.prepare('''
INSERT INTO compras_claras.prioridades_por_corte_departamento
(corte, departamento, valor_base, id_proceso, entidad,
 noticias_entidad, nivel_menciones, estado_revision, url_secop, criterio)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
''')

CORTE_CLASE = date(2026, 9, 3)

# 77 escrituras síncronas: claridad didáctica; NO es benchmark de carga masiva.
for _, f in candidatos.iterrows():
    session.execute(insertar, (
        CORTE_CLASE,
        texto_seguro(f.get("departamento_entidad")),
        int(f["precio_base"]),
        str(f["id_del_proceso"]),
        str(f["entidad"]),
        int(f["noticias_entidad"]),
        str(f["nivel_menciones"]),
        "pendiente",
        texto_seguro(f.get("urlproceso"), ""),
        "entidad en prensa; contratación directa; 0 respuestas",
    ))
print("Insertadas/actualizadas:", len(candidatos), "filas del corte", CORTE_CLASE)

In [ ]:
consulta = '''
SELECT id_proceso, entidad, valor_base, noticias_entidad, nivel_menciones, estado_revision
FROM compras_claras.prioridades_por_corte_departamento
WHERE corte = %s AND departamento = %s
LIMIT 5
'''

bogota = "Distrito Capital de Bogotá"
top5 = list(session.execute(consulta, (CORTE_CLASE, bogota)))
print("Top 5 —", bogota)
for f in top5:
    print(
        f.id_proceso, "|", f.entidad, "| $", f"{f.valor_base:,}",
        "| prensa:", f.noticias_entidad, f.nivel_menciones,
        "|", f.estado_revision,
    )
if not top5:
    print("No aparecieron filas. Revisa el nombre exacto del departamento.")

### EVIDENCIA INDIVIDUAL S05 — compara dos motores

Ya fijaste el top 5 esperado con pandas. Ahora pregunta lo mismo a Cassandra. La evidencia es una prueba de **corrección**, no un benchmark de velocidad.

In [ ]:
# 🧠 ENTIENDE · verifica que pandas y Cassandra respondan la misma pregunta.
if "departamento_elegido" not in globals() or "ids_esperados_pd" not in globals():
    raise RuntimeError("Ejecuta primero CONTRATO DE RESULTADO S05.")

top5_propio = list(session.execute(consulta, (CORTE_CLASE, departamento_elegido)))
ids_cql = [str(f.id_proceso) for f in top5_propio]
coinciden_cql_pd = ids_cql == ids_esperados_pd

print("Departamento     :", departamento_elegido)
print("Esperado pandas :", ids_esperados_pd)
print("Devuelto CQL    :", ids_cql)
print("¿Coinciden?     :", "SÍ" if coinciden_cql_pd else "NO")

if not coinciden_cql_pd:
    raise AssertionError("Revisa corte, departamento, carga y orden: los dos resultados no coinciden.")

### INTERPRETACIÓN CQL S05

**Cómo se lee.** Comparamos, en orden, los IDs del top 5 calculado con pandas y el servido por Cassandra.

**Qué nos dice.** Si coinciden, la tabla query-first está sirviendo correctamente esa pregunta sobre los datos cargados.

**Qué NO permite concluir todavía.** No demuestra que Cassandra sea más rápido ni necesario para 77 filas; no hicimos una prueba de rendimiento o escala.

**Error frecuente.** Convertir una prueba de corrección en una afirmación de performance.

**PARA LLEVAR.** Cambiar de motor no debería cambiar la decisión de negocio. pandas fija el contrato; Cassandra debe servir la misma respuesta para esta consulta.

In [ ]:
# UPDATE: cambiamos el estado y después lo volvemos a leer.
if top5:
    objetivo = top5[0]
    session.execute('''
    UPDATE compras_claras.prioridades_por_corte_departamento
    SET estado_revision = %s
    WHERE corte = %s AND departamento = %s AND valor_base = %s AND id_proceso = %s
    ''', ("en_revision", CORTE_CLASE, bogota, int(objetivo.valor_base), objetivo.id_proceso))

    verificacion = session.execute('''
    SELECT estado_revision
    FROM compras_claras.prioridades_por_corte_departamento
    WHERE corte = %s AND departamento = %s AND valor_base = %s AND id_proceso = %s
    ''', (CORTE_CLASE, bogota, int(objetivo.valor_base), objetivo.id_proceso)).one()

    assert verificacion is not None and verificacion.estado_revision == "en_revision"
    print("UPDATE verificado:", objetivo.id_proceso, "→", verificacion.estado_revision)

In [ ]:
# DELETE sin destruir la bandeja: creamos una fila centinela y luego la borramos.
demo_insert = session.prepare('''
INSERT INTO compras_claras.prioridades_por_corte_departamento
(corte, departamento, valor_base, id_proceso, entidad, estado_revision, url_secop, criterio)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
''')

session.execute(demo_insert, (
    CORTE_CLASE, "DEMO", 1, "S05-DEMO",
    "FILA DE PRACTICA", "pendiente", "", "solo para practicar DELETE"
))

antes = list(session.execute('''
SELECT id_proceso FROM compras_claras.prioridades_por_corte_departamento
WHERE corte = %s AND departamento = %s
''', (CORTE_CLASE, "DEMO")))

session.execute('''
DELETE FROM compras_claras.prioridades_por_corte_departamento
WHERE corte = %s AND departamento = %s AND valor_base = %s AND id_proceso = %s
''', (CORTE_CLASE, "DEMO", 1, "S05-DEMO"))

despues = list(session.execute('''
SELECT id_proceso FROM compras_claras.prioridades_por_corte_departamento
WHERE corte = %s AND departamento = %s
''', (CORTE_CLASE, "DEMO")))

print("Antes del DELETE:", [x.id_proceso for x in antes])
print("Después del DELETE:", [x.id_proceso for x in despues])

### El error intencional

Prueba mentalmente esta consulta:

```sql
SELECT *
FROM compras_claras.prioridades_por_corte_departamento
WHERE entidad = 'MINISTERIO DEL DEPORTE';
```

La columna **sí existe**, pero la tabla no fue diseñada para localizar particiones por `entidad`.

Ese es el aprendizaje:

> **Que una columna exista no significa que este modelo esté preparado para buscar por ella.**

Si “buscar por entidad” se vuelve una consulta profesional importante, Cassandra suele pedir **otra tabla diseñada para ese patrón de acceso**, no confiar en un recorrido global.

---
<details>
<summary><strong>MÁS ADELANTE — Consistencia ajustable</strong></summary>

Cassandra replica datos. ¿Cuántas confirmaciones esperamos antes de responder? Esa decisión intercambia latencia y garantía inmediata.

No configuramos niveles hoy: primero debe quedar firme `consulta → partición → clustering`. Astra Serverless además aplica guardrails propios.
</details>

---
## 11. MongoDB Atlas y Cassandra: no compiten por el mismo papel

| Necesidad en Compras Claras | Motor de hoy | Razón |
|---|---|---|
| noticias con estructura flexible | MongoDB Atlas | documento irregular y exploración |
| clasificar y publicar una vista | MongoDB Atlas | aggregation pipeline + vista |
| cruzar con SECOP y construir la regla | pandas | integración analítica explícita |
| servir repetidamente `corte + departamento → top 5` | Cassandra/Astra | tabla modelada para ese acceso |

**PARA LLEVAR.** Cassandra no “confirma” que los 77 sean urgentes. Solo sirve de forma eficiente una priorización que ya decidiste antes.

### HOJA DE TRUCOS S05 — para consultar sin volver veinte celdas

| Necesidad | Recuerda |
|---|---|
| filtrar documentos en Atlas | `Documents` + filtro |
| encadenar transformaciones | `Aggregations` + pipeline |
| conservar la receta | `Save → Save as` |
| publicar resultado consultable | `Save → Create view` |
| localizar datos en Cassandra | primero la **partition key** |
| ordenar dentro de la partición | **clustering columns** |
| consulta de Laura | `WHERE corte = ? AND departamento = ? LIMIT 5` |
| nueva consulta por entidad | probablemente **otra tabla**, no `ALLOW FILTERING` como parche |

**PARA LLEVAR.** MongoDB favorece exploración documental flexible. Cassandra favorece patrones de acceso conocidos y repetitivos.

### CHECKPOINT TRES PRODUCTOS S05 — antes de generar el hito

| Producto | Señal de que está listo |
|---|---|
| **1 · Vista Atlas** | existe `menciones_clasificadas` y controlas `6 + 25 + 111 = 142` |
| **2 · Bandeja explicable** | puedes reconstruir `1.000 → 163 → 77` y explicar qué refutó H1 |
| **3 · Consulta operacional** | sabes por qué la PK sirve `corte + departamento → top 5`, ejecutaste CQL y viste la misma tabla desde Python |

El hito y el ancla S6 que siguen **documentan** este trabajo. No agregan un cuarto o quinto objetivo.

---
## 12. Hito de la sesión

El hito captura **tu ejecución y tu decisión**, no memoria de sintaxis.

La celda siguiente crea:

- `s05_priorizacion.csv`;
- `hito_s05_servicio_prioridades.md`.

Te preguntará dos cosas que una IA no puede sacar de la nada sin conocer tu ejecución:
qué alternativa descartaste y qué consulta importante no soporta bien tu tabla.

---
## PUENTE S05-S06 — elige la fila que Laura abrirá después

Hasta aquí S5 respondió **qué mirar primero**. La próxima sesión no vuelve a construir esa decisión: toma **uno de tus procesos priorizados** y pregunta **qué relaciones existen alrededor de él**.

Si ejecutaste el contrato individual, verás tu top de pandas y escogerás cuál proceso llevar. Esa pequeña decisión hace que S6 empiece desde tu propia ejecución y no desde una fila impuesta por el cuaderno.

**Qué debe verse si salió bien:** un JSON con el ID, entidad, NIT, departamento, valor, contexto de prensa y criterio de priorización.  
**Error probable:** elegir un número fuera de la lista. Significa que la selección no corresponde a tu top disponible.  
**Recuperación:** vuelve a ejecutar la celda y elige uno de los números mostrados; si no existe el top individual, la celda usa el primer candidato como respaldo explícito.

In [ ]:
import json

if "top5_esperado_pd" in globals() and len(top5_esperado_pd):
    opciones_ancla = top5_esperado_pd.reset_index(drop=True)
    print("Elige el proceso que quieres llevar a S6:")
    for i, fila in opciones_ancla.iterrows():
        print(
            f"{i+1:>2}. {fila['id_del_proceso']} | {fila['entidad']} | "
            f"$ {float(fila['precio_base']):,.0f}"
        )
    seleccion_ancla = int(input("Número de proceso para S6: ").strip())
    if not 1 <= seleccion_ancla <= len(opciones_ancla):
        raise ValueError("El número debe corresponder a uno de los procesos mostrados.")
    fila_ancla = opciones_ancla.iloc[seleccion_ancla - 1]
    origen_eleccion_s06 = "selección propia dentro del top pandas S05"
else:
    fila_ancla = candidatos.iloc[0]
    seleccion_ancla = 1
    origen_eleccion_s06 = "respaldo: primer candidato de la bandeja S05"
    print("No existe top individual en memoria; se usa el primer candidato como respaldo.")

ancla_s06 = {
    "id_proceso": str(fila_ancla["id_del_proceso"]),
    "referencia": str(fila_ancla.get("referencia_del_proceso", "")),
    "entidad": str(fila_ancla["entidad"]),
    "nit_entidad": str(fila_ancla.get("nit_entidad", "")),
    "departamento": str(fila_ancla.get("departamento_entidad", "")),
    "valor_base": int(float(fila_ancla["precio_base"])),
    "modalidad": str(fila_ancla.get("modalidad_de_contratacion", "")),
    "noticias_entidad": int(fila_ancla["noticias_entidad"]),
    "nivel_menciones": str(fila_ancla["nivel_menciones"]),
    "url_secop": str(fila_ancla.get("urlproceso", "")),
    "criterio_priorizacion": "entidad en prensa; contratación directa; 0 respuestas",
    "alcance_prensa": "contexto a nivel de entidad; sin referencia exacta del proceso en título/subtítulo",
    "hipotesis_h1": "refutada: ninguna referencia SECOP exacta apareció en título/subtítulo",
    "origen": "bandeja operacional S05: 1.000→163→77",
    "origen_eleccion": origen_eleccion_s06,
}

with open("s05_ancla_s06.json", "w", encoding="utf-8") as f:
    json.dump(ancla_s06, f, ensure_ascii=False, indent=2)

print("Ancla S6 lista:")
print(json.dumps(ancla_s06, ensure_ascii=False, indent=2))

try:
    from google.colab import files
    files.download("s05_ancla_s06.json")
except Exception:
    print("Archivo guardado como s05_ancla_s06.json")

### Interpretación del ancla elegida

**Cómo se lee.** El JSON conserva una fila que ya pasó la regla `1.000→163→77` y registra si fue una elección propia o un respaldo.

**Qué nos dice.** S6 puede comenzar desde un proceso concreto sin rehacer la priorización.

**Qué NO permite concluir todavía.** Elegir una fila para profundizar no significa que sea irregular ni la “más riesgosa”. Faltan relaciones contractuales históricas y evidencia específica del proceso.

**Error frecuente.** Tratar la posición en la bandeja como una probabilidad de fraude.

In [ ]:
alternativa = input("Alternativa de diseño descartada: ").strip()
razon_alternativa = input("¿Por qué la descartaste para la consulta de Laura?: ").strip()
consulta_no_soportada = input("Una consulta profesional que esta tabla NO soporta bien: ").strip()
nueva_particion = input("Si fuera frecuente, ¿qué dato(s) usarías para localizar la nueva partición?: ").strip()

departamento_hito = globals().get("departamento_elegido", "No seleccionado")
ids_pd_hito = globals().get("ids_esperados_pd", [])
ids_cql_hito = globals().get("ids_cql", [])
coincidencia_hito = globals().get("coinciden_cql_pd", False)
primer_id = str(candidatos.iloc[0]["id_del_proceso"])
primer_entidad = str(candidatos.iloc[0]["entidad"])
primer_valor = int(candidatos.iloc[0]["precio_base"])
primer_noticias = int(candidatos.iloc[0]["noticias_entidad"])
primer_nivel = str(candidatos.iloc[0]["nivel_menciones"])

candidatos.to_csv("s05_priorizacion.csv", index=False, encoding="utf-8")

hito = f'''# Hito S05 — De la priorización al servicio

## Resultado propio de Atlas
- Fuente: {"vista real de Atlas" if vista_real else "respaldo; falta evidencia de vista real"}
- Alta / media / baja: {niveles.get("alta", 0)} / {niveles.get("media", 0)} / {niveles.get("baja", 0)}

## Regla de priorización
- Procesos iniciales: {len(secop)}
- Coincidencias por entidad: {len(paso1)}
- Candidatos: {len(candidatos)}
- Primer candidato: {primer_id} — {primer_entidad} — $ {primer_valor:,}
- Contexto desde Atlas: {primer_noticias} noticias — nivel {primer_nivel}

## Contraste de hipótesis de prensa
- H1: al menos una de las 77 referencias SECOP exactas aparece en título/subtítulo.
- Operacionalización: búsqueda literal de las 77 referencias exactas.
- Resultado observado: {con_referencia}/77 coincidencias exactas.
- Decisión sobre H1: {"refutada bajo esta prueba literal" if con_referencia == 0 else "no refutada por esta prueba literal"}.
- Alcance de la prensa: contexto a nivel de entidad; falta todavía evidencia textual específica del proceso.
- Hipótesis siguiente: relación temática o relacional mediante objeto, proveedor, fechas, cuerpo completo o relevancia textual.

## Límite
Referencias de proceso citadas literalmente en prensa: {con_referencia} de {len(candidatos)}.
La evidencia periodística usada es por entidad; no demuestra irregularidad del contrato específico.

## Query-first
PRIMARY KEY ((corte, departamento), valor_base, id_proceso)
CLUSTERING ORDER BY (valor_base DESC, id_proceso ASC)

## Evidencia individual de corrección
- Departamento: {departamento_hito}
- Top esperado con pandas: {ids_pd_hito}
- Top devuelto por Cassandra: {ids_cql_hito if ids_cql_hito else "NO VERIFICADO EN ASTRA"}
- Coincidencia exacta de IDs y orden: {"sí" if coincidencia_hito else "no verificada"}

## Alternativa descartada
{alternativa or "PENDIENTE"}

Razón: {razon_alternativa or "PENDIENTE"}

## Consulta no soportada
{consulta_no_soportada or "PENDIENTE"}

Nueva localización de partición si fuera frecuente: {nueva_particion or "PENDIENTE"}

## Decisión
MongoDB transforma documentos; pandas materializa una regla auditable; Cassandra sirve una proyección para una consulta repetitiva conocida.
'''

with open("hito_s05_servicio_prioridades.md", "w", encoding="utf-8") as f:
    f.write(hito)
print(hito)
print("
Archivos creados: s05_priorizacion.csv, hito_s05_servicio_prioridades.md")

### CONTRASTE DE HIPÓTESIS S05 — usar la prensa para preguntar mejor, no para acusar

Hasta ahora sabemos que las noticias aportan **contexto sobre entidades**. Antes de convertir ese contexto en una afirmación sobre contratos concretos, formulamos una hipótesis de trabajo que pueda fallar de forma observable.

> **Hipótesis de trabajo H1:** al menos una de las 77 referencias SECOP exactas aparece literalmente en los títulos o subtítulos examinados del corpus de prensa.

Esta H1 es útil pedagógicamente porque no depende de opiniones: sabemos exactamente qué observar para sostenerla o refutarla.

```text
77 referencias SECOP exactas
        ↓
búsqueda literal en título + subtítulo
        ↓
¿aparece al menos una?
```

**Resultado del contraste:** `0/77` coincidencias exactas. En este corpus y bajo esta definición literal, **H1 queda refutada**.

Eso **no es un test estadístico inferencial**: no hay p-valor ni estimación poblacional. Es un **contraste empírico de una hipótesis de trabajo** sobre datos concretos y una regla de observación concreta.

#### ¿Qué decisión permite tomar?

El resultado no invalida la prensa; obliga a **especificar mejor qué evidencia aporta**:

```text
ANTES, demasiado fuerte:
"la prensa respalda estos procesos"

DESPUÉS, mejor especificada:
"la prensa aporta contexto sobre las entidades;
 todavía falta evidencia textual específica del proceso"
```

Aquí la prensa **gana especificidad analítica** porque ahora conocemos con precisión su unidad de evidencia:

- **unidad observada en prensa:** la entidad y su intensidad de mención;
- **unidad que Laura revisa:** el proceso contractual;
- **puente actual:** contexto de entidad, no identificación literal del proceso.

**Cómo se lee.** Ninguna de las 77 referencias exactas apareció en los títulos o subtítulos examinados.  
**Qué nos dice.** La prensa sigue siendo útil para **formular y refinar hipótesis**, priorizar contexto y decidir qué evidencia adicional buscar.  
**Qué NO permite concluir todavía.** No demuestra que ninguna noticia sea relevante para esos procesos: no examinamos aquí cuerpos completos, nombres alternativos, objeto contractual, proveedor, fechas ni similitud semántica.  
**Error frecuente.** Confundir “H1 literal fue refutada” con “la prensa no sirve”. El contraste justamente nos dice **para qué sí sirve y hasta dónde llega**.

**PARA LLEVAR.** Una buena analítica no busca confirmar la primera historia; formula una hipótesis falsable, define cómo observarla, mira la evidencia y **reformula la afirmación al nivel que los datos soportan**.

### CONTRASTE DE HIPÓTESIS S05 — usar la prensa para preguntar mejor, no para acusar

Hasta ahora sabemos que las noticias aportan **contexto sobre entidades**. Antes de convertir ese contexto en una afirmación sobre contratos concretos, formulamos una hipótesis de trabajo que pueda fallar de forma observable.

> **Hipótesis de trabajo H1:** al menos una de las 77 referencias SECOP exactas aparece literalmente en los títulos o subtítulos examinados del corpus de prensa.

Esta H1 es útil pedagógicamente porque no depende de opiniones: sabemos exactamente qué observar para sostenerla o refutarla.

```text
77 referencias SECOP exactas
        ↓
búsqueda literal en título + subtítulo
        ↓
¿aparece al menos una?
```

**Resultado del contraste:** `0/77` coincidencias exactas. En este corpus y bajo esta definición literal, **H1 queda refutada**.

Eso **no es un test estadístico inferencial**: no hay p-valor ni estimación poblacional. Es un **contraste empírico de una hipótesis de trabajo** sobre datos concretos y una regla de observación concreta.

#### ¿Qué decisión permite tomar?

El resultado no invalida la prensa; obliga a **especificar mejor qué evidencia aporta**:

```text
ANTES, demasiado fuerte:
"la prensa respalda estos procesos"

DESPUÉS, mejor especificada:
"la prensa aporta contexto sobre las entidades;
 todavía falta evidencia textual específica del proceso"
```

Aquí la prensa **gana especificidad analítica** porque ahora conocemos con precisión su unidad de evidencia:

- **unidad observada en prensa:** la entidad y su intensidad de mención;
- **unidad que Laura revisa:** el proceso contractual;
- **puente actual:** contexto de entidad, no identificación literal del proceso.

**Cómo se lee.** Ninguna de las 77 referencias exactas apareció en los títulos o subtítulos examinados.  
**Qué nos dice.** La prensa sigue siendo útil para **formular y refinar hipótesis**, priorizar contexto y decidir qué evidencia adicional buscar.  
**Qué NO permite concluir todavía.** No demuestra que ninguna noticia sea relevante para esos procesos: no examinamos aquí cuerpos completos, nombres alternativos, objeto contractual, proveedor, fechas ni similitud semántica.  
**Error frecuente.** Confundir “H1 literal fue refutada” con “la prensa no sirve”. El contraste justamente nos dice **para qué sí sirve y hasta dónde llega**.

**PARA LLEVAR.** Una buena analítica no busca confirmar la primera historia; formula una hipótesis falsable, define cómo observarla, mira la evidencia y **reformula la afirmación al nivel que los datos soportan**.

### CONTRASTE DE HIPÓTESIS S05 — usar la prensa para preguntar mejor, no para acusar

Hasta ahora sabemos que las noticias aportan **contexto sobre entidades**. Antes de convertir ese contexto en una afirmación sobre contratos concretos, formulamos una hipótesis de trabajo que pueda fallar de forma observable.

> **Hipótesis de trabajo H1:** al menos una de las 77 referencias SECOP exactas aparece literalmente en los títulos o subtítulos examinados del corpus de prensa.

Esta H1 es útil pedagógicamente porque no depende de opiniones: sabemos exactamente qué observar para sostenerla o refutarla.

```text
77 referencias SECOP exactas
        ↓
búsqueda literal en título + subtítulo
        ↓
¿aparece al menos una?
```

**Resultado del contraste:** `0/77` coincidencias exactas. En este corpus y bajo esta definición literal, **H1 queda refutada**.

Eso **no es un test estadístico inferencial**: no hay p-valor ni estimación poblacional. Es un **contraste empírico de una hipótesis de trabajo** sobre datos concretos y una regla de observación concreta.

#### ¿Qué decisión permite tomar?

El resultado no invalida la prensa; obliga a **especificar mejor qué evidencia aporta**:

```text
ANTES, demasiado fuerte:
"la prensa respalda estos procesos"

DESPUÉS, mejor especificada:
"la prensa aporta contexto sobre las entidades;
 todavía falta evidencia textual específica del proceso"
```

Aquí la prensa **gana especificidad analítica** porque ahora conocemos con precisión su unidad de evidencia:

- **unidad observada en prensa:** la entidad y su intensidad de mención;
- **unidad que Laura revisa:** el proceso contractual;
- **puente actual:** contexto de entidad, no identificación literal del proceso.

**Cómo se lee.** Ninguna de las 77 referencias exactas apareció en los títulos o subtítulos examinados.  
**Qué nos dice.** La prensa sigue siendo útil para **formular y refinar hipótesis**, priorizar contexto y decidir qué evidencia adicional buscar.  
**Qué NO permite concluir todavía.** No demuestra que ninguna noticia sea relevante para esos procesos: no examinamos aquí cuerpos completos, nombres alternativos, objeto contractual, proveedor, fechas ni similitud semántica.  
**Error frecuente.** Confundir “H1 literal fue refutada” con “la prensa no sirve”. El contraste justamente nos dice **para qué sí sirve y hasta dónde llega**.

**PARA LLEVAR.** Una buena analítica no busca confirmar la primera historia; formula una hipótesis falsable, define cómo observarla, mira la evidencia y **reformula la afirmación al nivel que los datos soportan**.

### CONTRASTE DE HIPÓTESIS S05 — usar la prensa para preguntar mejor, no para acusar

Hasta ahora sabemos que las noticias aportan **contexto sobre entidades**. Antes de convertir ese contexto en una afirmación sobre contratos concretos, formulamos una hipótesis de trabajo que pueda fallar de forma observable.

> **Hipótesis de trabajo H1:** al menos una de las 77 referencias SECOP exactas aparece literalmente en los títulos o subtítulos examinados del corpus de prensa.

Esta H1 es útil pedagógicamente porque no depende de opiniones: sabemos exactamente qué observar para sostenerla o refutarla.

```text
77 referencias SECOP exactas
        ↓
búsqueda literal en título + subtítulo
        ↓
¿aparece al menos una?
```

**Resultado del contraste:** `0/77` coincidencias exactas. En este corpus y bajo esta definición literal, **H1 queda refutada**.

Eso **no es un test estadístico inferencial**: no hay p-valor ni estimación poblacional. Es un **contraste empírico de una hipótesis de trabajo** sobre datos concretos y una regla de observación concreta.

#### ¿Qué decisión permite tomar?

El resultado no invalida la prensa; obliga a **especificar mejor qué evidencia aporta**:

```text
ANTES, demasiado fuerte:
"la prensa respalda estos procesos"

DESPUÉS, mejor especificada:
"la prensa aporta contexto sobre las entidades;
 todavía falta evidencia textual específica del proceso"
```

Aquí la prensa **gana especificidad analítica** porque ahora conocemos con precisión su unidad de evidencia:

- **unidad observada en prensa:** la entidad y su intensidad de mención;
- **unidad que Laura revisa:** el proceso contractual;
- **puente actual:** contexto de entidad, no identificación literal del proceso.

**Cómo se lee.** Ninguna de las 77 referencias exactas apareció en los títulos o subtítulos examinados.  
**Qué nos dice.** La prensa sigue siendo útil para **formular y refinar hipótesis**, priorizar contexto y decidir qué evidencia adicional buscar.  
**Qué NO permite concluir todavía.** No demuestra que ninguna noticia sea relevante para esos procesos: no examinamos aquí cuerpos completos, nombres alternativos, objeto contractual, proveedor, fechas ni similitud semántica.  
**Error frecuente.** Confundir “H1 literal fue refutada” con “la prensa no sirve”. El contraste justamente nos dice **para qué sí sirve y hasta dónde llega**.

**PARA LLEVAR.** Una buena analítica no busca confirmar la primera historia; formula una hipótesis falsable, define cómo observarla, mira la evidencia y **reformula la afirmación al nivel que los datos soportan**.

### HIPÓTESIS REVISADA S05 — ¿qué evidencia buscaríamos después?

El `0/77` no cierra la investigación: **mejora la siguiente pregunta**.

> **H2:** aunque el ID exacto no aparezca, algunos procesos podrían estar relacionados temáticamente con noticias mediante su entidad, objeto, proveedor, lugar o periodo.

Para evaluar H2 necesitaríamos evidencia más específica, por ejemplo:

| Evidencia adicional | Qué pregunta permitiría hacer |
|---|---|
| nombre/objeto del procedimiento | ¿la noticia habla del mismo tema contractual? |
| proveedor | ¿el actor aparece en otras relaciones relevantes? |
| fechas | ¿la noticia y el proceso son temporalmente compatibles? |
| cuerpo completo del texto | ¿hay menciones que no aparecen en título/subtítulo? |
| búsqueda por relevancia | ¿qué documentos son más pertinentes aunque no compartan el ID literal? |

Esto prepara dos pasos del semestre:

```text
S6 → relaciones: entidad → proceso → proveedor
S7 → texto: relevancia, no solo coincidencia literal
```

**Pregunta de negocio.** Laura no necesita que la prensa “condene” un contrato. Necesita que le ayude a **formular mejores hipótesis de revisión y pedir la siguiente evidencia correcta**.

In [ ]:
try:
    from google.colab import files
    files.download("hito_s05_servicio_prioridades.md")
    files.download("s05_priorizacion.csv")
except ImportError:
    print("Fuera de Colab: descarga los archivos desde el explorador de tu entorno.")

In [ ]:
#@title Cerrar conexiones { display-mode: "form" }
# CERRAR CONEXIONES S05
# Buena práctica: cerrar clientes al terminar la sesión.
try:
    cluster.shutdown()
    print("Conexión Cassandra cerrada.")
except Exception:
    pass

try:
    if "client" in globals():
        client.close()
        print("Conexión MongoDB cerrada.")
except Exception:
    pass

## Rúbrica de calidad del hito

| Criterio | Completo | Parcial | Sin evidencia | Peso |
|---|---|---|---|---:|
| Vista Atlas | vista real + 142 / 6-25-111 | respaldo declarado | números sin fuente | 15 |
| Regla + contraste | 1.000→163→77 + H1 + operacionalización + conclusión correcta | reporta 0/77 sin explicar qué hipótesis probó | convierte prensa o bandeja en acusación | 20 |
| Query-first | PK/clustering explicados desde la consulta | copia diseño sin justificar | clave no sirve esa consulta | 20 |
| Evidencia individual | departamento + top pandas + top CQL + coincidencia | top pandas y contingencia Astra declarada | no hay resultado propio | 20 |
| Alternativa | alternativa + razón de descarte | alternativa sin razón | no hay alternativa | 15 |
| Consulta no soportada | consulta distinta + nueva localización | consulta sin rediseño | afirma que cualquier filtro funciona | 10 |

Las autoevaluaciones son **formativas**. La evidencia revisable es el hito producido por la ejecución.

# CIERRE PEDAGÓGICO S05 — la pregunta de S4 por fin tiene una respuesta operacional

S4 terminó con **datos persistidos y compartidos**. S5 convirtió ese estado en una cadena defendible:

```text
142 entidades en Atlas
        ↓
6 alta / 25 media / 111 baja
        ↓
1.000 procesos SECOP
        ↓
163 con contexto de prensa disponible
        ↓
77 candidatos bajo una heurística explícita
        ↓
CONTRASTE DE H1
¿aparece al menos una referencia SECOP exacta en título/subtítulo?
        ↓
0/77 → H1 literal refutada
        ↓
la prensa queda mejor especificada:
contexto de entidad, no evidencia directa del proceso
        ↓
top esperado con pandas
        ↓
misma respuesta servida por Cassandra
        ↓
un proceso elegido como ancla para S6
```

El valor pedagógico del `0/77` **no es terminar en cero**. Es mostrar una disciplina analítica:

```text
formular hipótesis
→ definir qué observación la puede refutar
→ mirar la evidencia
→ reducir o reformular la afirmación
→ pedir la siguiente evidencia correcta
```

**PARA LLEVAR.** Laura ya puede explicar por qué un proceso llegó a su bandeja, qué papel cumplió la prensa y qué NO puede afirmar todavía. La prensa sirve para **dar contexto, formular hipótesis y orientar la siguiente búsqueda de evidencia**; no convierte una mención de entidad en acusación sobre un contrato.

Lo más importante de Cassandra hoy tampoco fue la sintaxis CQL: fue comprobar que **el diseño de almacenamiento nació de una pregunta concreta** y que el nuevo servicio devolvió la misma respuesta que la lógica analítica que lo alimentó.

## Lo que sigue — una hipótesis refutada produce una pregunta mejor

S5 no terminó diciendo “la prensa no sirve”. Terminó diciendo algo mucho más preciso:

> **La prensa aporta contexto de entidad; la prueba literal no encontró IDs de proceso en título/subtítulo.**

Por eso la siguiente hipótesis ya no es “¿aparece el ID exacto?”, sino:

> **H2: ¿existe evidencia específica alrededor de este proceso mediante actores, relaciones o contenido temático?**

La próxima sesión empieza cuando Laura abre **el proceso que acabas de elegir** y pregunta:

> **“Ya sé por qué este proceso llegó a mi bandeja y cuál es el alcance real de la prensa. Antes de asignarlo a un auditor, ¿qué relaciones alrededor de su entidad y sus procesos históricos necesito ver?”**

El candidato de S5 será el **ancla**. Los procesos históricos adjudicados aportarán hechos que el candidato todavía no tiene: proveedores, otras contrataciones y conexiones con otras entidades.

```text
S5  contexto de prensa + bandeja + H1 literal refutada
 ↓
S6  especificidad relacional: entidad → proceso → proveedor
 ↓
S7  especificidad textual: relevancia, no solo coincidencia literal
```

Ahí aparece Neo4j. No para declarar irregularidades, sino para **probar una nueva hipótesis con evidencia relacional**.